In [3]:
import numpy as np
import pandas as pd
from scipy.stats import linregress, pearsonr

from armored.models import *
from armored.preprocessing import *

import itertools

from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn import tree

In [16]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")
df_exp4a = pd.read_csv("data/exp4/exp4_metabolites_best_reps.csv")
df_exp4b = pd.read_csv("data/exp4/exp4_metabolites_new_best.csv")
df_exp4c = pd.read_csv("data/exp4/exp4_metabolites_new_worst.csv")
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3, df_exp4a, df_exp4b, df_exp4c))

# define variable names
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

# average replicates
df_fmt_mean = []
for exp_name, df_exp in df.groupby("Experiments"):
    df_groups = df_exp.groupby("Time")
    df_avg = df_groups[system_variables].mean().reset_index()
    df_avg.insert(0, "Experiments", [exp_name]*df_avg.shape[0])
    df_fmt_mean.append(df_avg)
df = pd.concat(df_fmt_mean)

In [4]:
# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(df)
df_scaled = scaler.transform(df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
data = format_data(df.copy(), species, metabolites, controls, observed=observed)
data_scaled = format_data(df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), n_metabolites=len(metabolites), n_controls=len(controls), n_hidden=32)
# fit model
brnn.fit(data_scaled, evd_tol=1e-3)

Total measurements: 27517, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1429.134, Residuals: -0.00633
Loss: 1345.974, Residuals: -0.00327
Loss: 1316.323, Residuals: -0.00082
Loss: 1285.769, Residuals: -0.00132
Loss: 1170.579, Residuals: 0.00147
Loss: 1144.611, Residuals: -0.00382
Loss: 1098.094, Residuals: -0.00274
Loss: 1011.266, Residuals: -0.00266
Loss: 929.189, Residuals: -0.00162
Loss: 886.026, Residuals: -0.00242
Loss: 840.357, Residuals: -0.00421
Loss: 828.633, Residuals: -0.00429
Loss: 818.472, Residuals: -0.00198
Loss: 801.120, Residuals: -0.00273
Loss: 777.498, Residuals: -0.00269
Loss: 775.382, Residuals: -0.00308
Loss: 754.145, Residuals: -0.00309
Loss: 714.870, Residuals: -0.00285
Loss: 714.203, Residuals: -0.00284
Loss: 686.828, Residuals: -0.00293
Loss: 658.182, Residuals: -0.00346
Loss: 656.110, Residuals: -0.00311
Loss: 636.528, Residuals: -0.00279
Loss: 603.775, Residuals: -0.00223
Loss: 602.632, Residuals: -0.00234
Loss: 591.798, Residuals: -0.0

In [5]:
# log that ignores zeros
def zlog(x):
    return jnp.where(x > 0, jnp.log(x), 0.)

# shannon diversity
def shannon(y):
    y_rel = y / jnp.sum(y)
    return jnp.nansum(-zlog(y_rel)*y_rel)

# determine best previously observed values of objectives
b_max = 24.206144013
d_max = 2.256914760709628 
v_max = 0.05319052969911115

# define objective 
def objective(y):
    
    # endpoint shannon diversity
    diversity = shannon(y[-1, :len(species)]) / d_max
    
    # variance in species abundances in last two passages
    species_stdv = jnp.std(y[-2:, :len(species)], 0)
    instability  = jnp.where(species_stdv>0, species_stdv, 0).mean() / v_max
    
    # endpoint butyrate production 
    butyrate = y[-1, -2] / b_max 
    
    return diversity - instability + butyrate

# function handle evaluates objective in batches
objective_batch = vmap(objective)

In [6]:
# save predictions to dataframe
pred_dfs = []
for T, X, U, Y, exp_names in data_scaled: 
    
    # set Y as a copy
    Y = np.copy(Y)
    
    # keep initial condition
    P = np.array(brnn.predict_point(X, U))
    
    # unscale measurements
    for eval_time in scaler.eval_times:
        t_inds = T == eval_time
        Y[t_inds] *= (scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"])
        Y[t_inds] += scaler.scale_dict_obs[f"{eval_time} min"]
    
    # unscale predictions
    for eval_time in scaler.eval_times:
        t_inds = T == eval_time
        P[t_inds] *= (scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"])
        P[t_inds] += scaler.scale_dict_obs[f"{eval_time} min"]
    
    # get shapes
    batch_size, n_time, n_out = P.shape
    
    # measured objective
    obj_meas = objective_batch(Y)
    obj_pred = objective_batch(P)
    
    # reshape arrays
    T = np.reshape(T, batch_size*n_time)
    U = np.reshape(U, [batch_size*n_time, U.shape[-1]])
    Y = np.reshape(Y, [batch_size*n_time, n_out])
    P = np.reshape(P, [batch_size*n_time, n_out])
    exp_names_array = np.reshape(np.array([np.tile(item, n_time) for item in exp_names]), n_time * len(exp_names))
    
    # names of variables
    outputs = species+metabolites
    outputs_meas = [o + " meas" for o in outputs]
    outputs_pred = [o + " pred" for o in outputs]
    
    # save to df
    pred_df = pd.DataFrame()
    pred_df['Experiments'] = exp_names_array
    pred_df['Time'] = T
    pred_df['Objective meas'] = np.repeat(obj_meas, n_time)
    pred_df['Objective pred'] = np.repeat(obj_pred, n_time)
    pred_df[outputs_meas] = Y
    pred_df[outputs_pred] = P
    pred_df[controls] = U
    pred_dfs.append(pred_df)
    
pred_df = pd.concat(pred_dfs)
pred_df.to_csv("space/sampled_space.csv", index=False)

In [8]:
# full factorial of both species and controls 
n_species = len(species)
n_fibers  = len(controls)

# create matrix of all possible communities
Slist = [np.reshape(np.array(i), (1, n_species)) for i in itertools.product([0, 1], repeat = n_species)]
# remove all zeros community
S = np.array(np.concatenate(Slist)[1:], float)

# create matrix of all possible fiber combinations
Flist = [np.reshape(np.array(i), (1, n_fibers)) for i in itertools.product([0, 1], repeat = n_fibers)]
F = np.array(np.concatenate(Flist), float)

# matrix of species and fiber indeces 
M = np.array(np.zeros([S.shape[0]*F.shape[0], 2]), int)
k = 0 
for i in range(S.shape[0]):
    for j in range(F.shape[0]):
        M[k, 0] = int(i)
        M[k, 1] = int(j)
        k += 1

# function to pull sample data 
def gen_exp_cond(k):
    s_ind, f_ind = M[k]
    return S[s_ind], F[f_ind]

# function to generate informative name of experimental condition
def gen_exp_name(Si, Fi):
    exp_name = ""
    for i,si in enumerate(Si):
        if si > 0:
            exp_name += species[i].split("abs")[0] + "-"
    for i,fi in enumerate(Fi):
        if fi > 0:
            exp_name += controls[i] + "-"
    return exp_name[:-1]

In [20]:
def format_design_data(S, F, t_eval, scaler, batch_size=512):

    # data is a list of tuples (T, X, U, Y, names) where each tuple corresponds to a batch
    data = []
    
    # total number of samples
    n_samples = S.shape[0] * F.shape[0]
    k = 0 
    
    # number of evaluation times
    n_eval = len(t_eval)

    # divide data into batches
    for batch_inds in tqdm(np.array_split(np.arange(n_samples), np.ceil(n_samples / batch_size))):

        # initialize data matrices 
        T = np.empty([len(batch_inds), n_eval])
        T[:] = t_eval
        X = np.empty([len(batch_inds), S.shape[-1]+len(metabolites)])
        X[:] = np.nan
        U = np.empty([len(batch_inds), n_eval, F.shape[-1]])
        U[:] = np.nan

        # keep track of experiment names
        names = []
        for i, batch_ind in enumerate(batch_inds):

            # pull sample data 
            Si, Fi = gen_exp_cond(k)

            # keep track of experiment names
            names.append(gen_exp_name(Si, Fi))

            # store initial condition data
            X[i, :len(Si)] = .000667 * Si
            
            # initial pH and metabolites
            X[i, len(Si):] = np.array([6.7, 0., 0., 0.])
            
            # scale observed variables 
            X[i] = (X[i] - scaler.scale_dict_obs["0 min"]) / scaler.scale_dict_obs["0 max"]
            
            # store controls (already scaled)
            if sum(Fi)>0:
                U[i] =  Fi / sum(Fi)
            else:
                U[i] = Fi
            
            # update counter
            k += 1

        data.append((T, X, U, names))

    return data

In [21]:
# format and scale data based on training data
design_data = format_design_data(S, F, np.array([0, 1, 2, 3]), scaler, batch_size=1024)

100%|███████████████████████████████████████| 2048/2048 [00:21<00:00, 95.30it/s]


In [24]:
# save predictions to dataframe
pred_dfs = []
# for T, X, U, Y, exp_names in data_scaled: 
for T, X, U, exp_names in design_data:
    
    # keep initial condition
    preds = np.array(brnn.predict_point(X, U))
    
    # unscale predictions
    for eval_time in scaler.eval_times:
        t_inds = T == eval_time
        preds[t_inds] *= (scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"])
        preds[t_inds] += scaler.scale_dict_obs[f"{eval_time} min"]
    
    # get shapes
    batch_size, n_time, n_out = preds.shape
    
    # reshape arrays
    T = np.reshape(T, batch_size*n_time)
    U = np.reshape(U, [batch_size*n_time, U.shape[-1]])
    Y = np.reshape(preds, [batch_size*n_time, n_out])
    exp_names_array = np.reshape(np.array([np.tile(item, n_time) for item in exp_names]), n_time * len(exp_names))
    
    # save to df
    pred_df = pd.DataFrame()
    pred_df['Experiments'] = exp_names_array
    pred_df['Time'] = T
    pred_df[species+metabolites] = Y
    pred_df[controls] = U
    pred_dfs.append(pred_df)
    
pred_df = pd.concat(pred_dfs)

In [25]:
# all predicted variables
y = pred_df[observed].values

# reshape into [n_samples, n_passages, n_variables]
y_batch = np.reshape(y, [y.shape[0]//4, 4, y.shape[-1]])

# evaluate obective of every sample
obj_batch = objective_batch(y_batch)

# store objective for all time points
pred_df['Objective'] = np.repeat(obj_batch, 4)

In [26]:
# save to csv
pred_df.to_csv("space/space.csv", index=False)

In [27]:
pred_df

,Experiments,Time,ACabs,BAabs,BHabs,BLabs,BUabs,CAabs,CCabs,CHabs,...,Lactate,Butyrate,Acetate,AcGum,ArGal,Inulin,Pectin,Starch,Xylan,Objective
0,RI,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049654
1,RI,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.042092,1.906625,14.540738,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049654
2,RI,2.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049654
3,RI,3.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.669601,1.656765,6.417598,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049654
4,RI-Xylan,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.335008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4087,AC-BA-BH-BL-BU-CA-CC-CH-DF-EL-ER-FP-PC-PJ-RI-A...,3.0,0.022939,0.193776,0.000000,0.000000,0.182286,0.000000,0.136966,0.000000,...,0.240772,8.001605,22.746085,0.200000,0.200000,0.200000,0.200000,0.200000,0.000000,0.858180
4088,AC-BA-BH-BL-BU-CA-CC-CH-DF-EL-ER-FP-PC-PJ-RI-A...,0.0,0.000667,0.000667,0.000667,0.000667,0.000667,0.000667,0.000667,0.000667,...,0.000000,0.000000,0.000000,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.910289
4089,AC-BA-BH-BL-BU-CA-CC-CH-DF-EL-ER-FP-PC-PJ-RI-A...,1.0,0.034305,0.118248,0.006939,0.016604,0.309078,0.000000,0.067389,0.013777,...,0.681623,10.654917,16.179780,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.910289
4090,AC-BA-BH-BL-BU-CA-CC-CH-DF-EL-ER-FP-PC-PJ-RI-A...,2.0,0.025429,0.158405,0.000000,0.000000,0.283465,0.000000,0.190207,0.000000,...,NaN,NaN,NaN,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.910289
